# DPD — Revisión de archivos

Carga dos archivos Excel (schedule + payment tape), calcula DPD en ambos modos y exporta resultados.

**Parámetros a editar:** `SCHEDULE_PATH`, `PT_PATH`, `CALC_DATE`, `GRACE_DAYS` en la primera celda.

Después corré las celdas en orden.

In [30]:
import os
import sys
from datetime import date

import pandas as pd

sys.path.insert(0, os.path.abspath('.'))
from dpd.excel_runner import run_from_excel, export_results

# ═══════════════════════════════════════════════════════
# PARÁMETROS — editar acá antes de correr
# ═══════════════════════════════════════════════════════

SCHEDULE_PATH    = 'tests/Days Past Due.xlsx'   # Excel con scheduled_payments_installments
PT_PATH          = 'tests/Days Past Due.xlsx'   # Excel con Payment_Tape (puede ser el mismo)
SCHEDULE_SHEET   = None   # None = autodetect por nombre de hoja
PT_SHEET         = None

CALC_DATE        = date(2026, 10, 3)   # ← FECHA DE CORTE
MODE             = 'both'              # 'cascade' | 'join' | 'both'
GRACE_DAYS       = 1                   # días calendario de gracia tras vencimiento
PARTIAL_COUNTS   = False               # solo aplica en cascade

OUT_PATH         = 'resultado_dpd.xlsx'

# ═══════════════════════════════════════════════════════

print(f'Schedule:   {SCHEDULE_PATH}')
print(f'PT:         {PT_PATH}')
print(f'Corte:      {CALC_DATE}')
print(f'Grace days: {GRACE_DAYS}')
print(f'Modo:       {MODE}')

In [31]:
# === Cómputo ===
# run_from_excel carga, sanitiza y calcula en un solo paso.
result_df, summary_df = run_from_excel(
    schedule_path    = SCHEDULE_PATH,
    payment_tape_path= PT_PATH,
    calc_date        = CALC_DATE,
    mode             = MODE,
    grace_days       = GRACE_DAYS,
    partial_counts   = PARTIAL_COUNTS,
    schedule_sheet   = SCHEDULE_SHEET,
    payment_tape_sheet = PT_SHEET,
)

In [32]:
# === Clasificador de casos (1-18) ===
# Cada regla detecta un patrón específico. Permitimos múltiples matches (B2):
# si los datos son indistinguibles entre casos (ej. 3 y 17), se devuelven todos.
# Si no calza con ninguno → ['Sin clasificar'].
#
# Caso 14 (Reestructurado) NO se puede detectar sin metadata extra (status/version).

def _enrich_contract(sched_c, pay_c, calc_date):
    """sched ordenado por fecha + total_paid_by_ref, n_pagos_ref, overdue."""
    sched_c = sched_c.copy().sort_values("date").reset_index(drop=True)
    if len(pay_c) > 0:
        agg = (pay_c.groupby("borrower_installment_reference", as_index=False)
                    .agg(total_paid_by_ref=("total_payment", "sum"),
                         n_pagos_ref=("total_payment", "count")))
    else:
        agg = pd.DataFrame(columns=["borrower_installment_reference", "total_paid_by_ref", "n_pagos_ref"])
    sched_c = sched_c.merge(agg, on="borrower_installment_reference", how="left")
    sched_c["total_paid_by_ref"] = sched_c["total_paid_by_ref"].fillna(0).astype(float)
    sched_c["n_pagos_ref"] = sched_c["n_pagos_ref"].fillna(0).astype(int)
    sched_c["overdue"] = sched_c["date"] <= calc_date
    return sched_c


def _has_late_pay(sched_c, pay_c):
    if len(pay_c) == 0:
        return False
    merged = pay_c.merge(
        sched_c[["borrower_installment_reference", "date"]],
        on="borrower_installment_reference", how="left",
    )
    return ((merged["payment_date"] > merged["date"]) & merged["date"].notna()).any()


def _has_early_pay(sched_c, pay_c):
    if len(pay_c) == 0:
        return False
    merged = pay_c.merge(
        sched_c[["borrower_installment_reference", "date"]],
        on="borrower_installment_reference", how="left",
    )
    return ((merged["payment_date"] < merged["date"]) & merged["date"].notna()).any()


def classify_contract(sched_c, pay_c, calc_date):
    """Lista de casos (1-18) que matchean. Vacío → ['Sin clasificar']."""
    n_pagos = len(pay_c)
    matches = []

    enriched = _enrich_contract(sched_c, pay_c, calc_date)
    overdue = enriched[enriched["overdue"]].reset_index(drop=True)
    n_overdue = len(overdue)

    if n_overdue == 0:
        return ["Sin clasificar"]

    # Casos 3 y 17: sin pagos. Indistinguibles por datos.
    if n_pagos == 0:
        return [3, 17]

    paid_exact = (overdue["total_paid_by_ref"] - overdue["gross_amount"]).abs() < 0.001
    partial = (overdue["total_paid_by_ref"] > 0) & (overdue["total_paid_by_ref"] + 0.001 < overdue["gross_amount"])
    impaga = overdue["total_paid_by_ref"] == 0
    sobrepagada = overdue["total_paid_by_ref"] > overdue["gross_amount"] + 0.001
    covered = overdue["total_paid_by_ref"] >= overdue["gross_amount"] - 0.001  # paid_exact OR sobrepagada

    n_paid = paid_exact.sum()
    n_partial = partial.sum()
    n_impaga = impaga.sum()
    n_sobre = sobrepagada.sum()
    n_covered = covered.sum()

    has_late = _has_late_pay(enriched, pay_c)
    has_early = _has_early_pay(enriched, pay_c)
    has_pay_on_corte = (pay_c["payment_date"] == calc_date).any()

    diff = overdue["gross_amount"] - overdue["total_paid_by_ref"]
    has_centavos = ((diff > 0.001) & (diff < 1.0)).any()

    # Cuota con 2 pagos en el mismo mes calendario que sumen gross
    has_doble_pago_mismo_mes = False
    for ref in overdue["borrower_installment_reference"].unique():
        ref_pays = pay_c[pay_c["borrower_installment_reference"] == ref]
        if len(ref_pays) == 2:
            cuota_gross = float(overdue.loc[overdue["borrower_installment_reference"] == ref, "gross_amount"].iloc[0])
            if abs(ref_pays["total_payment"].sum() - cuota_gross) < 0.01:
                d1, d2 = list(ref_pays["payment_date"])
                if d1.year == d2.year and d1.month == d2.month:
                    has_doble_pago_mismo_mes = True
                    break

    # ─── Reglas exclusivas ──────────────────────────────────────────────────

    # Caso 1: Happy path estricto
    if (n_paid == n_overdue and not has_late and not has_early and n_sobre == 0):
        matches.append(1)

    # Caso 2: Última cuota vencida es la única parcial
    if (n_overdue >= 2 and n_partial == 1 and n_impaga == 0 and n_sobre == 0
            and partial.iloc[-1] and paid_exact.iloc[:-1].all()):
        matches.append(2)

    # Caso 4: Cubiertas con pago tardío, una cuota con sobrepago modesto OK (intereses moratorios).
    # Excluye caso 10 (doble pago mismo mes) y caso 12 (pago el día del corte).
    if (n_covered == n_overdue and has_late and not has_early
            and (overdue["n_pagos_ref"] <= 1).all()
            and not has_doble_pago_mismo_mes
            and not has_pay_on_corte):
        matches.append(4)

    # Caso 5: Una cuota intermedia parcial sin recuperar (ni la última, ni centavos)
    if (n_overdue >= 3 and n_partial == 1 and n_impaga == 0 and n_sobre == 0
            and not partial.iloc[-1] and (paid_exact | partial).all() and not has_centavos):
        matches.append(5)

    # Caso 6: Cubiertas con pago anticipado, sin tardío
    if (n_paid == n_overdue and has_early and not has_late and n_sobre == 0):
        matches.append(6)

    # Caso 7: Sobrepago modesto en exactamente una cuota (1.001 ≤ ratio ≤ 1.4),
    # sin tardío, sin temprano, sin pago en corte
    if n_sobre == 1 and n_partial == 0 and n_impaga == 0:
        ratio = (overdue["total_paid_by_ref"] / overdue["gross_amount"]).max()
        if 1.001 <= ratio <= 1.4 and not has_late and not has_early and not has_pay_on_corte:
            matches.append(7)

    # Caso 8: 2+ cuotas con pago parcial
    if n_partial >= 2:
        matches.append(8)

    # Caso 9: Sobrepago ~1.5x en una cuota Y la siguiente cubre menos que gross
    for i in range(n_overdue - 1):
        gross_i = overdue["gross_amount"].iloc[i]
        if gross_i > 0:
            rat_i = overdue["total_paid_by_ref"].iloc[i] / gross_i
            if 1.4 <= rat_i <= 1.6:
                next_paid = overdue["total_paid_by_ref"].iloc[i + 1]
                next_gross = overdue["gross_amount"].iloc[i + 1]
                if next_paid + 0.001 < next_gross:
                    matches.append(9)
                    break

    # Caso 10: Cuota con 2 pagos del mismo mes que sumen gross
    if has_doble_pago_mismo_mes:
        matches.append(10)

    # Caso 11: Una sola cuota pagada (la primera), resto impagas
    if n_paid == 1 and n_impaga >= 2 and n_partial == 0 and paid_exact.iloc[0]:
        matches.append(11)

    # Caso 12: Algún pago el día del corte
    if has_pay_on_corte:
        matches.append(12)

    # Caso 13: Diferencia de centavos en alguna cuota
    if has_centavos:
        matches.append(13)

    # Caso 14: Reestructurado — no detectable sin metadata. Skip.

    # Caso 15: Cuota con sobrepago ≥ 1.9x (duplicado)
    ratios = overdue["total_paid_by_ref"] / overdue["gross_amount"].replace(0, 1)
    if (ratios >= 1.9).any():
        matches.append(15)

    # Caso 16: Primera cuota vencida impaga, resto cubiertas exactas
    if (n_overdue >= 2 and impaga.iloc[0] and paid_exact.iloc[1:].all()
            and n_partial == 0 and n_sobre == 0):
        matches.append(16)

    # Caso 18: Pagos consecutivos al inicio + impagas consecutivas al final, sin tardío
    if (n_paid >= 2 and n_impaga >= 1 and n_partial == 0 and n_sobre == 0 and not has_late):
        first_unpaid_idx = next((i for i, p in enumerate(paid_exact.values) if not p), None)
        if first_unpaid_idx is not None and first_unpaid_idx >= 2:
            before = paid_exact.iloc[:first_unpaid_idx]
            after_impaga = impaga.iloc[first_unpaid_idx:]
            if before.all() and after_impaga.all():
                matches.append(18)

    return matches if matches else ["Sin clasificar"]


def caso_str(matches):
    if not matches:
        return "Sin clasificar"
    return ", ".join(str(m) for m in matches)


print("Clasificador definido.")

Clasificador definido.


In [33]:
# === Clasificación de contratos (18 casos del catálogo de prueba) ===
# Requiere la hoja 'Casos' en el Excel de schedule.
from IPython.display import display

try:
    cases_df = pd.read_excel(SCHEDULE_PATH, sheet_name='Casos')
    cases_df = cases_df.rename(columns={
        '#': 'caso', 'Caso': 'titulo',
        'Borrower contract id': 'borrower_contract_id',
        'Descripción': 'descripcion',
        'Resultado esperado del script': 'esperado',
    })
    has_cases = True
except Exception:
    print('⚠  No se encontró la hoja Casos — omitiendo clasificación.')
    has_cases = False

# Cargar sched_df y pay_df para el clasificador (necesita los DataFrames originales)
from dpd.excel_runner import load_schedule, load_payment_tape
sched_df = load_schedule(SCHEDULE_PATH, SCHEDULE_SHEET)
pay_df   = load_payment_tape(PT_PATH, PT_SHEET)

caso_por_contrato = {}
if has_cases:
    for cid, sched_c in sched_df.groupby('borrower_contract_id'):
        pay_c = pay_df[pay_df['borrower_contract_id'] == cid]
        try:
            matches = classify_contract(sched_c, pay_c, CALC_DATE)
            caso_por_contrato[cid] = caso_str(matches)
        except Exception:
            caso_por_contrato[cid] = 'Sin clasificar'
    result_df['CASE'] = result_df['borrower_contract_id'].map(caso_por_contrato)
    summary_df['case'] = summary_df['borrower_contract_id'].map(caso_por_contrato)

print(f'Contratos clasificados: {len(caso_por_contrato)}')

In [34]:
# === Resumen por contrato ===
display(summary_df)

In [35]:
# === Detalle por cuota ===
with pd.option_context('display.max_rows', 50):
    display(result_df)

In [36]:
# === Exportar a Excel ===
export_results(result_df, summary_df, OUT_PATH)